In [5]:
import os

# Import necessary libraries

from google.adk.models.lite_llm import LiteLlm # For OpenAI support


# Convenience libraries for working with Neo4j inside of Google ADK
from neo4j_for_adk import graphdb, tool_success, tool_error

from typing import Dict, Any
from dotenv import load_dotenv

import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)

print("Libraries imported.")

Libraries imported.


In [6]:
load_dotenv()

MODEL_NAME = os.getenv("DEEPSEEK_MODEL")

llm = LiteLlm(MODEL_NAME)

print(llm.llm_client.completion(
    model=llm.model,
    messages=[{"role": "user", "content": "你准备好了吗？"}],
    tools=[],
))

print("\n Deepseek已经准备好了")

ModelResponse(id='0bc0cf4e-0357-40f2-8ffd-c85c0dd70094', created=1788854803, model='deepseek-v4-flash', object='chat.completion', system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', choices=[Choices(finish_reason='stop', index=0, message=Message(content='准备好了！随时可以为你解答问题或提供帮助。你想聊什么呢？', role='assistant', tool_calls=None, function_call=None, reasoning_content='我们需要回答用户。用户问“你准备好了吗？” 可能是开始对话。需要回应友好，表示准备好，并询问需要什么帮助。简洁。', provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=47, prompt_tokens=87, total_tokens=134, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=31, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=87))

 Deepseek已经准备好了


In [7]:
neo4j_is_ready = graphdb.send_query("RETURN 'Neo4j is Ready!' as message")

print(neo4j_is_ready)

{'status': 'success', 'query_result': [{'message': 'Neo4j is Ready!'}]}


In [8]:
def create_uniqueness_constraint(
    label: str,
    unique_property_key: str,
) -> Dict[str, Any]:
    """为一个节点标签（label）和它的属性键（property key）创建“唯一性约束”。

    唯一性约束可以确保：没有任何两个具有相同标签的节点，会在该属性上拥有相同的值。
    这能极大地提高数据导入和后续查询的【性能】与【数据完整性】。

    参数 (Args):
        label: 要创建约束的节点标签（比如 "Product" 产品）。
        unique_property_key: 必须具有唯一值的属性名（比如 "id" 或 "name"）。

    返回 (Returns):
        返回一个包含状态键（'success' 成功 或 'error' 错误）的字典。
        如果发生错误，字典中会包含一个 'error_message'（错误信息）键。
    """

    # 【注意】：这里必须使用 Python 的 f-string（字符串格式化）。
    # 因为 Neo4j 的 Cypher 语法在“创建约束”时，不支持将标签和属性名作为参数（parameterization）传入。
    constraint_name = f"{label}_{unique_property_key}_constraint"

    # 拼装 Cypher 查询语句
    # 意思是：如果没有的话，就创建一个名叫 constraint_name 的约束。
    # 针对所有标签为 label 的节点 n，要求它的 unique_property_key 属性必须是唯一的（UNIQUE）。
    query = f"""CREATE CONSTRAINT `{constraint_name}` IF NOT EXISTS
    FOR (n:`{label}`)
    REQUIRE n.`{unique_property_key}` IS UNIQUE"""

    # 调用 graphdb (图数据库连接对象) 发送执行这条语句
    results = graphdb.send_query(query)

    return results

In [26]:
def load_nodes_from_csv(
    source_file: str,
    label: str,
    unique_column_name: str,
    properties: list[str],
) -> Dict[str, Any]:
    """从 CSV 文件中批量加载节点"""

    # 从 CSV 文件中加载节点，
    # 并根据 unique_column_name 指定的唯一列值进行合并
    query = f"""LOAD CSV WITH HEADERS FROM "file:///" + $source_file AS row
    CALL (row) {{
        MERGE (n:$($label) {{ {unique_column_name} : row[$unique_column_name] }})
        FOREACH (k IN $properties | SET n[k] = row[k])
    }} IN TRANSACTIONS OF 1000 ROWS
    """

    # 将参数传递给 Neo4j 图数据库并执行查询
    results = graphdb.send_query(query, {
        "source_file": source_file,
        "label": label,
        "unique_column_name": unique_column_name,
        "properties": properties
    })

    return results



In [12]:
def import_nodes(node_construction: dict) -> dict:
    """根据节点构建规则导入节点。"""

    # 为 unique_column 创建唯一性约束
    uniqueness_result = create_uniqueness_constraint(
        node_construction["label"],
        node_construction["unique_column_name"]
    )

    # 如果创建唯一性约束失败，则直接返回错误结果
    if uniqueness_result["status"] == "error":
        return uniqueness_result

    # 从 CSV 文件中导入节点
    load_nodes_result = load_nodes_from_csv(
        node_construction["source_file"],
        node_construction["label"],
        node_construction["unique_column_name"],
        node_construction["properties"]
    )

    return load_nodes_result

In [25]:
def import_relationships(relationship_construction: dict) -> Dict[str, Any]:
    """根据关系构建规则导入关系。"""

    # 从关系构建规则中获取起始节点和目标节点对应的 CSV 列名
    from_node_column = relationship_construction["from_node_column"]
    to_node_column = relationship_construction["to_node_column"]

    query = f"""LOAD CSV WITH HEADERS FROM "file:///" + $source_file AS row
    CALL (row) {{
        MATCH (from_node:$($from_node_label) {{ {from_node_column} : row[$from_node_column] }}),
              (to_node:$($to_node_label) {{ {to_node_column} : row[$to_node_column] }} )
        MERGE (from_node)-[r:$($relationship_type)]->(to_node)
        FOREACH (k IN $properties | SET r[k] = row[k])
    }} IN TRANSACTIONS OF 1000 ROWS
    """

    # 将关系构建规则中的参数传递给 Neo4j 图数据库并执行查询
    results = graphdb.send_query(query, {
        "source_file": relationship_construction["source_file"],
        "from_node_label": relationship_construction["from_node_label"],
        "from_node_column": relationship_construction["from_node_column"],
        "to_node_label": relationship_construction["to_node_label"],
        "to_node_column": relationship_construction["to_node_column"],
        "relationship_type": relationship_construction["relationship_type"],
        "properties": relationship_construction["properties"]
    })

    return results

In [16]:
def construct_domain_graph(construction_plan: dict) -> Dict[str, Any]:

    node_constructions = [
        value
        for value in construction_plan.values()
        if value['construction_type'] == 'node'
    ]

    for node_construction in node_constructions:
        result = import_nodes(node_construction)

        if result["status"] == "error":
            return result

    relationship_constructions = [
        value
        for value in construction_plan.values()
        if value['construction_type'] == 'relationship'
    ]

    for relationship_construction in relationship_constructions:
        result = import_relationships(relationship_construction)

        if result["status"] == "error":
            return result

    return {
        "status": "success"
    }

In [23]:
approved_construction_plan = {
    "Assembly": {
        "construction_type": "node",
        "source_file": "bom/assemblies.csv",
        "label": "Assembly",
        "unique_column_name": "assembly_id",
        "properties": ["assembly_name", "quantity", "product_id"]
    },

    "Part": {
        "construction_type": "node",
        "source_file": "bom/components.csv",
        "label": "Part",
        "unique_column_name": "part_id",
        "properties": ["part_name", "quantity", "assembly_id"]
    },

    "Contains": {
        "construction_type": "relationship",
        "source_file": "bom/assemblies.csv",
        "relationship_type": "Contains",
        "from_node_label": "Product",
        "from_node_column": "product_id",
        "to_node_label": "Assembly",
        "to_node_column": "assembly_id",
        "properties": ["quantity"]
    },

    "Is_Part_Of": {
        "construction_type": "relationship",
        "source_file": "bom/components.csv",
        "relationship_type": "Is_Part_Of",
        "from_node_label": "Part",
        "from_node_column": "part_id",
        "to_node_label": "Assembly",
        "to_node_column": "assembly_id",
        "properties": ["quantity"]
    },

    "Supplied_By": {
        "construction_type": "relationship",
        "source_file": "bom/part_supplier_mapping.csv",
        "relationship_type": "Supplied_By",
        "from_node_label": "Part",
        "from_node_column": "part_id",
        "to_node_label": "Supplier",
        "to_node_column": "supplier_id",
        "properties": [
            "supplier_name",
            "lead_time_days",
            "unit_cost",
            "minimum_order_quantity",
            "preferred_supplier"
        ]
    }
}

In [27]:
construct_domain_graph(approved_construction_plan)

{'status': 'success'}

In [28]:
# 提取关系构建规则列表
relationship_constructions = [
    value for value in approved_construction_plan.values()
    if value.get("construction_type") == "relationship"
]

relationship_constructions

[{'construction_type': 'relationship',
  'source_file': 'bom/assemblies.csv',
  'relationship_type': 'Contains',
  'from_node_label': 'Product',
  'from_node_column': 'product_id',
  'to_node_label': 'Assembly',
  'to_node_column': 'assembly_id',
  'properties': ['quantity']},
 {'construction_type': 'relationship',
  'source_file': 'bom/components.csv',
  'relationship_type': 'Is_Part_Of',
  'from_node_label': 'Part',
  'from_node_column': 'part_id',
  'to_node_label': 'Assembly',
  'to_node_column': 'assembly_id',
  'properties': ['quantity']},
 {'construction_type': 'relationship',
  'source_file': 'bom/part_supplier_mapping.csv',
  'relationship_type': 'Supplied_By',
  'from_node_label': 'Part',
  'from_node_column': 'part_id',
  'to_node_label': 'Supplier',
  'to_node_column': 'supplier_id',
  'properties': ['supplier_name',
   'lead_time_days',
   'unit_cost',
   'minimum_order_quantity',
   'preferred_supplier']}]

In [29]:
# 一个比较巧妙的 Cypher 查询：
# 用来展示每一条关系构建规则对应的一个实际关系实例

# 将关系构建规则列表转换成多条单独处理的规则
unwind_list = "UNWIND $relationship_constructions AS construction"

# 根据指定的 construction.relationship_type 匹配一条关系路径
# 最终只返回路径中三个部分的标签和关系类型：
# 起始节点标签、关系类型、目标节点标签
match_one_path = """
    MATCH (from)-[r:$(construction.relationship_type)]->(to)
    RETURN labels(from) AS fromNode, type(r) AS relationship, labels(to) AS toNode
    LIMIT 1
"""

# 将上面的 MATCH 查询放入子查询中，
# 使每一条 construction 都可以单独执行一次
match_in_subquery = f"""
CALL (construction) {{
{match_one_path}
}}
"""

# 将前面的查询片段组合成完整的 Cypher 查询
cypher = f"""
{unwind_list}
{match_in_subquery}
RETURN fromNode, relationship, toNode
"""

# 打印最终生成的 Cypher 查询，方便检查
print(cypher)

print("\n---")

# 执行 Cypher 查询，并将关系构建规则作为参数传递给 Neo4j
graphdb.send_query(cypher, {
    "relationship_constructions": relationship_constructions
})


UNWIND $relationship_constructions AS construction

CALL (construction) {

    MATCH (from)-[r:$(construction.relationship_type)]->(to)
    RETURN labels(from) AS fromNode, type(r) AS relationship, labels(to) AS toNode
    LIMIT 1

}

RETURN fromNode, relationship, toNode


---


{'status': 'success',
 'query_result': [{'fromNode': ['Part'],
   'relationship': 'Is_Part_Of',
   'toNode': ['Assembly']}]}